In [1]:
import os

In [2]:
%pwd

'c:\\Users\\user\\Desktop\\Data Science Projects\\End-to-End-Kidney-Disease-Deep-Learning-Classification-Project-MLflow-DVC-Deployment\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\user\\Desktop\\Data Science Projects\\End-to-End-Kidney-Disease-Deep-Learning-Classification-Project-MLflow-DVC-Deployment'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories


In [7]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH):

            self.config = read_yaml(config_filepath)
            self.params = read_yaml(params_filepath)

            create_directories([self.config.artifacts_root])  

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config    

In [8]:
import os
import zipfile
import gdown
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size

In [9]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self) -> str:
        """
        Fetch data from the url 
        """
        try:
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs("artifacts/data_ingestion", exist_ok=True)
            logger.info(f"Downloading file from :[{dataset_url}] into :[{zip_download_dir}]")

            file_id = dataset_url.split('/')[-2]
            prefix = 'https://drive.google.com/uc?/export=download&id='
            gdown.download(prefix + file_id, str(self.config.local_data_file), quiet=False)
            logger.info(f"Downloaded data into file of size: {get_size(Path(self.config.local_data_file))}")

        except Exception as e:
            raise e

    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

Defining a pipeline

In [10]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()

    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-10-19 19:04:45,289: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-19 19:04:45,291: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-19 19:04:45,291: INFO: common: created directory at: artifacts]
[2026-10-19 19:04:45,297: INFO: common: created directory at: artifacts/data_ingestion]
[2026-10-19 19:04:45,299: INFO: 1403585679: Downloading file from :[https://drive.google.com/file/d/11GpFx5MIlHuiVo5qFUE07NBAUP3p2vR1/view?usp=sharing] into :[artifacts/data_ingestion/data.zip]]


Downloading...
From (original): https://drive.google.com/uc?id=11GpFx5MIlHuiVo5qFUE07NBAUP3p2vR1
From (redirected): https://drive.google.com/uc?id=11GpFx5MIlHuiVo5qFUE07NBAUP3p2vR1&confirm=t&uuid=05047997-e1dc-4126-b2a4-4612c1177084
To: c:\Users\user\Desktop\Data Science Projects\End-to-End-Kidney-Disease-Deep-Learning-Classification-Project-MLflow-DVC-Deployment\artifacts\data_ingestion\data.zip
100%|██████████| 938M/938M [01:35<00:00, 9.82MB/s] 


[2026-10-19 19:06:24,118: INFO: 1403585679: Downloaded data into file of size: ~ 916367 KB]
